# 3.6 — Blend v2 + submission

Mốc: ensemble v1 (`ml+dl_all` ở 3.4) — CV lồng 0.94852, public LB 0.94972.
Ứng viên: 9 model v1 + model mới của 3.5 (ML) và 3.3 vòng 4 (DL). Bản nhiều seed thay cho bản ít seed của cùng model
(vd. `lgb_nl3_x3` thay `lgb_nl3`, `mlp_plr_sig3_reg_x5` thay `..._x3`). Model chưa có output thì tự bỏ qua.

| Phương án | Mô tả |
|---|---|
| `v1` | Tái lập `ml+dl_all` của 3.4 (9 model, rank toàn cục, random-walk + shrink 0.3) |
| `rw_all` | Random-walk + shrink trên mọi model |
| `rw_all_foldrank` | Như trên, nhưng OOF được rank **trong từng fold** |
| `hill_all_foldrank` | Hill climbing (Caruana) trên rank trong từng fold |
| `stack_lr_all` | Logistic Regression trên logit xác suất của mọi model |

Chấm bằng CV lồng trên 5 fold chung. Chỉ thay `v1` khi phương án mới có AUC trung bình theo fold cao hơn **và thắng `v1` ở ≥ 4/5 fold**.

**Input:** competition + `ccf-shared-code` + output 3.0, 3.1, 3.3 (mới nhất), 3.5. **Accelerator:** None.
**Output:** `submission_v2.csv`, `3.ENSEMBLE/results/blend_comparison_v2.csv`, `final_blend_v2.json`.

In [ ]:
import sys
from pathlib import Path

# shared/ code: local repo or the Kaggle dataset that holds features.py + dl.py
cands = [Path("shared/features.py"), Path("../../shared/features.py"), *Path("/kaggle/input").rglob("features.py")]
feat = next(p for p in cands if p.is_file())
sys.path.insert(0, str(feat.parent.resolve()))
import features as F
print("shared code:", feat.parent)

In [ ]:
import time
import json
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

T0 = time.time()
def log(msg):
    print(f"[{(time.time() - T0) / 60:6.1f} min] {msg}", flush=True)

tr, te, y, train_id, test_id = F.load()
fold = F.load_folds(train_id)          # from notebook 3.0 output (folds_5_seed9.csv)
assert len(tr) == F.N_TRAIN and len(te) == F.N_TEST and set(np.unique(fold)) == set(range(F.N_FOLDS))
log(f"train {tr.shape}, test {te.shape}, fraud rate {y.mean():.4%}")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.preprocessing import StandardScaler

V1 = ["lgb_nl3", "lgb_nl4", "lgb_nl8", "spline_lr", "xgb",
      "mlp_plr_sig3_reg_x3", "mlp_plr_sig1_reg", "mlp_plr_sig3", "mlp_plr_sig10_reg"]
NEW = ["lgb_nl3_cnt", "lgb_nl3_x3", "lgb_nl4_x3", "catboost_d4", "catboost_d6", "catboost_d4_gpu", "catboost_d6_gpu",
       "mlp_plr_sig3_reg_x5", "mlp_plr_sig3_x3", "mlp_plr_sig10_reg_x3",
       "mlp_plr_sig2_reg", "mlp_plr_sig5_reg", "resnet_plr_sig3"]
REPLACES = {"lgb_nl3_x3": "lgb_nl3", "lgb_nl4_x3": "lgb_nl4", "mlp_plr_sig3_reg_x5": "mlp_plr_sig3_reg_x3",
            "mlp_plr_sig3_x3": "mlp_plr_sig3", "mlp_plr_sig10_reg_x3": "mlp_plr_sig10_reg"}

def available(n):
    try:
        F.find_prediction("oof", n)
        return True
    except FileNotFoundError:
        return False

missing_v1 = [n for n in V1 if not available(n)]
assert not missing_v1, f"Thiếu model v1: {missing_v1}"
new_avail = [n for n in NEW if available(n)]
print("Model mới có output:", new_avail)
print("Model mới thiếu:", [n for n in NEW if n not in new_avail])
replaced = {REPLACES[n] for n in new_avail if n in REPLACES}
ALL = [n for n in V1 if n not in replaced] + new_avail
NAMES = list(dict.fromkeys(V1 + ALL))
print("ALL:", ALL)

O, T = F.load_predictions(NAMES, train_id, test_id)
col = {n: i for i, n in enumerate(NAMES)}
R_glob = np.column_stack([F.rank01(O[:, i]) for i in range(len(NAMES))])
R_fold = np.column_stack([F.rank_within_folds(O[:, i], fold) for i in range(len(NAMES))])
R_test = np.column_stack([F.rank01(T[:, i]) for i in range(len(NAMES))])
single = pd.DataFrame({"oof_auc": [roc_auc_score(y, O[:, col[n]]) for n in NAMES],
                       "fold_auc_mean": [np.mean([roc_auc_score(y[fold == f], O[fold == f, col[n]])
                                                  for f in range(F.N_FOLDS)]) for n in NAMES]}, index=NAMES)
print(single.round(5))

In [ ]:
# ---------- phương án: fit(rows) -> (params, score(R_rows, O_rows))
def rank_blend(models, R, shrink=0.3, iters=500):
    idx = [col[m] for m in models]
    def fit(rows):
        w, _ = F.tune_weights(R[np.ix_(rows, idx)], y[rows], iters=iters)
        w = (1 - shrink) * w + shrink / len(idx)
        return {"weights": dict(zip(models, w.tolist()))}, lambda Rx, Ox: Rx[:, idx] @ w
    return fit

def hill(models, R, max_steps=40):
    idx = [col[m] for m in models]
    def fit(rows):
        w, _ = F.hill_climb(R[np.ix_(rows, idx)], y[rows], max_steps=max_steps)
        return {"weights": dict(zip(models, w.tolist()))}, lambda Rx, Ox: Rx[:, idx] @ w
    return fit

def logit(p):
    p = np.clip(p, 1e-7, 1 - 1e-7)
    return np.log(p / (1 - p))

def stack_lr(models, C=0.1):
    idx = [col[m] for m in models]
    def fit(rows):
        sc = StandardScaler().fit(logit(O[np.ix_(rows, idx)]))
        lr = LogisticRegression(C=C, max_iter=1000).fit(sc.transform(logit(O[np.ix_(rows, idx)])), y[rows])
        return ({"coef": dict(zip(models, lr.coef_[0].tolist()))},
                lambda Rx, Ox: lr.decision_function(sc.transform(logit(Ox[:, idx]))))
    return fit

# (fit, ma trận rank dùng để chấm OOF)
METHODS = {
    "v1": (rank_blend(V1, R_glob), R_glob),
    "rw_all": (rank_blend(ALL, R_glob), R_glob),
    "rw_all_foldrank": (rank_blend(ALL, R_fold), R_fold),
    "hill_all_foldrank": (hill(ALL, R_fold), R_fold),
    "stack_lr_all": (stack_lr(ALL), R_glob),
}

In [ ]:
# ---------- CV lồng
rows_out = []
for name, (fit, R) in METHODS.items():
    fold_auc, fold_ap = [], []
    for f, a, b in F.iter_folds(fold):
        _, score = fit(a)
        s = score(R[b], O[b])
        fold_auc.append(roc_auc_score(y[b], s))
        fold_ap.append(average_precision_score(y[b], s))
    rows_out.append({"method": name, "fold_auc_mean": np.mean(fold_auc), "fold_auc_std": np.std(fold_auc),
                     "fold_ap_mean": np.mean(fold_ap), "fold_auc": np.round(fold_auc, 5).tolist()})
    log(f"{name:18s} fold AUC mean={np.mean(fold_auc):.5f} ± {np.std(fold_auc):.5f} | AP={np.mean(fold_ap):.5f}")

cmp = pd.DataFrame(rows_out).set_index("method")
cmp["gain_vs_v1"] = cmp["fold_auc_mean"] - cmp.loc["v1", "fold_auc_mean"]
base = np.array(cmp.loc["v1", "fold_auc"])
cmp["folds_beating_v1"] = [int((np.array(v) > base).sum()) for v in cmp["fold_auc"]]
pd.set_option("display.width", 220)
print(cmp.round(5))
Path("3.ENSEMBLE/results").mkdir(parents=True, exist_ok=True)
cmp.to_csv("3.ENSEMBLE/results/blend_comparison_v2.csv")
single.to_csv("3.ENSEMBLE/results/single_models_v2.csv")

In [ ]:
# ---------- chọn phương án, học lại trên toàn OOF, áp lên test
best = cmp["fold_auc_mean"].idxmax()
if best != "v1" and cmp.loc[best, "folds_beating_v1"] < 4:
    print(f"{best} không thắng v1 ổn định ({cmp.loc[best, 'folds_beating_v1']}/5 fold) -> giữ v1")
    best = "v1"
print("Phương án được chọn:", best)

fit, R = METHODS[best]
params, score = fit(np.arange(len(y)))
test_score = score(R_test, T)
_, v1_score = METHODS["v1"][0](np.arange(len(y)))
v1_test = v1_score(R_test, T)

final = {"selected_method": best, "selection": "max nested-CV fold-mean ROC-AUC, must beat v1 on >= 4/5 folds",
         "nested_cv": cmp.loc[best, ["fold_auc_mean", "fold_auc_std", "fold_ap_mean", "gain_vs_v1"]].to_dict(),
         "fit_on_all_oof": params, "candidates": ALL, "leaderboard_used_for_selection": False}
Path("3.ENSEMBLE/results/final_blend_v2.json").write_text(json.dumps(final, indent=2, default=float))
print(json.dumps(final, indent=2, default=float))

In [ ]:
# ---------- submission
sample = pd.read_csv(F.find_file("sample_submission.csv"))
assert list(sample.columns) == ["id", "is_fraud"] and sample["id"].is_unique
assert set(sample["id"]) == set(test_id) and len(sample) == F.N_TEST

s = pd.Series(F.rank01(test_score), index=test_id)
sub = pd.DataFrame({"id": sample["id"], "is_fraud": s.reindex(sample["id"]).to_numpy()})
assert len(sub) == F.N_TEST and sub["is_fraud"].notna().all()
assert np.isfinite(sub["is_fraud"]).all() and sub["is_fraud"].between(0, 1).all()
sub.to_csv("submission_v2.csv", index=False)
print(f"saved submission_v2.csv: {len(sub):,} rows")
print("Spearman(test) v2 vs v1:", round(float(pd.Series(test_score).corr(pd.Series(v1_test), method="spearman")), 5))
log("done")